# Chest X-ray Pneumonia Detection (RSNA/Kaggle)

PyTorch + timm + Albumentations (MONAI optional)

## 2. Project Overview

Build a binary pneumonia classifier from chest X-rays using transfer learning and clinically cautious evaluation.

## 3. Learning Objectives

- Train a pneumonia classifier with transfer learning
- Evaluate with AUROC and confusion matrix
- Detect and prevent data leakage
- Document medical caveats responsibly

## 4. Problem Statement

Given a chest X-ray image, predict pneumonia presence (1) vs no pneumonia (0).

## 5. Why This Project Matters

Pneumonia triage support can improve workflow prioritization, but false negatives may delay critical care.

## 6. Dataset Overview

Target dataset: RSNA/Kaggle chest X-ray pneumonia data with patient-level metadata and image labels.

## 7. Dataset Source and License Notes

Use according to Kaggle/RSNA terms. Ensure compliance with medical data governance and access controls.

## 8. Environment Setup

Required: torch, timm, albumentations, scikit-learn, matplotlib, seaborn.
Optional: MONAI for medical imaging transforms/workflows.

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')

# MONAI optional
try:
    import monai
    HAS_MONAI = True
except Exception:
    HAS_MONAI = False

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('timm:', timm.__version__)
print('Albumentations:', A.__version__)
print('MONAI available:', HAS_MONAI)

In [ ]:
# 10. Configuration / constants
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
LR = 1e-4
EPOCHS_BASELINE = 1
EPOCHS_STRONG = 1

BASELINE_MODEL = 'resnet18'
STRONG_MODEL = 'efficientnet_b0'

SAVE_DIR = Path.cwd() / 'Computer Vision' / 'Chest X-ray Pneumonia Detection'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = SAVE_DIR / 'rsna_kaggle_data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('Save dir:', SAVE_DIR)

In [ ]:
# 11. Dataset loading
# Default synthetic mode for guaranteed execution in this environment.
FORCE_SYNTHETIC = True
use_synthetic = FORCE_SYNTHETIC

train_images, train_labels, train_patient_ids = [], [], []
val_images, val_labels, val_patient_ids = [], [], []
test_images, test_labels, test_patient_ids = [], [], []

if not use_synthetic:
    # Real-data expectation example:
    # - image files in stage_2_train_images
    # - labels CSV with patient/study ids and class labels
    # IMPORTANT: split by patient_id to avoid leakage across splits.
    pass

if use_synthetic:
    # Simulate patient-level dataset with mild class imbalance
    n_train = 4000
    n_val = 600
    n_test = 800

    for i in range(n_train):
        y = int(np.random.choice([0, 1], p=[0.62, 0.38]))
        train_labels.append(y)
        train_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
        train_patient_ids.append(f'train_{i:05d}')

    for i in range(n_val):
        y = int(np.random.choice([0, 1], p=[0.62, 0.38]))
        val_labels.append(y)
        val_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
        val_patient_ids.append(f'val_{i:05d}')

    for i in range(n_test):
        y = int(np.random.choice([0, 1], p=[0.62, 0.38]))
        test_labels.append(y)
        test_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
        test_patient_ids.append(f'test_{i:05d}')

print('Using synthetic mode:', use_synthetic)
print('Train/Val/Test sizes:', len(train_labels), len(val_labels), len(test_labels))

In [ ]:
# 12. Data validation checks + leakage sanity checks
assert len(train_images) == len(train_labels) == len(train_patient_ids)
assert len(val_images) == len(val_labels) == len(val_patient_ids)
assert len(test_images) == len(test_labels) == len(test_patient_ids)

# Leakage check: no patient ID overlap across splits
train_set = set(train_patient_ids)
val_set = set(val_patient_ids)
test_set = set(test_patient_ids)

ov_train_val = len(train_set.intersection(val_set))
ov_train_test = len(train_set.intersection(test_set))
ov_val_test = len(val_set.intersection(test_set))

print('Patient overlap train-val:', ov_train_val)
print('Patient overlap train-test:', ov_train_test)
print('Patient overlap val-test:', ov_val_test)

assert ov_train_val == 0
assert ov_train_test == 0
assert ov_val_test == 0
print('Validation checks passed.')

## 13. Exploratory Data Analysis

Inspect label distribution and sample chest X-rays.

In [ ]:
vals, cnts = np.unique(train_labels, return_counts=True)

plt.figure(figsize=(5,4))
plt.bar([str(v) for v in vals], cnts)
plt.title('Train Label Distribution (0=no pneumonia, 1=pneumonia)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(10, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(train_images[i], cmap='gray')
    ax.set_title(f'label={train_labels[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 14. Train/Validation/Test Split Strategy

Critical rule: split by patient ID, not image-level random split, to prevent leakage and inflated AUROC.

In [ ]:
# 15. Preprocessing and augmentation strategy
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.08, rotate_limit=8, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.3),
    A.GaussNoise(var_limit=(3, 15), p=0.15),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

print('Albumentations pipelines configured.')
if HAS_MONAI:
    print('MONAI is available for optional medical transform extension.')

## 16. Baseline Approach

Baseline transfer learning model: ResNet-18 binary classifier.

In [ ]:
class XrayDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        y = float(self.labels[idx])
        x = self.transform(image=img)['image']
        return x, torch.tensor(y, dtype=torch.float32)

train_ds = XrayDataset(train_images, train_labels, train_tf)
val_ds = XrayDataset(val_images, val_labels, val_tf)
test_ds = XrayDataset(test_images, test_labels, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

baseline_model = timm.create_model(BASELINE_MODEL, pretrained=True, num_classes=1).to(DEVICE)
strong_model = timm.create_model(STRONG_MODEL, pretrained=True, num_classes=1).to(DEVICE)

print('Baseline model:', BASELINE_MODEL)
print('Strong model:', STRONG_MODEL)
print('Data loaders ready.')

## 17. Main Model/Workflow

Stronger transfer-learning model: EfficientNet-B0.

In [ ]:
# 18. Training loop / fine-tuning pipeline
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0.0
    ys, probs = [], []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item())
        p = torch.sigmoid(logits)
        ys.extend(yb.cpu().numpy().astype(int).tolist())
        probs.extend(p.detach().cpu().numpy().tolist())

    preds = [1 if p >= 0.5 else 0 for p in probs]
    avg_loss = total_loss / max(len(loader), 1)
    acc = accuracy_score(ys, preds)
    f1 = f1_score(ys, preds, zero_division=0)
    rec = recall_score(ys, preds, zero_division=0)
    try:
        auroc = roc_auc_score(ys, probs)
    except Exception:
        auroc = float('nan')

    return avg_loss, acc, f1, rec, auroc, ys, probs


def train_model(model, name, epochs):
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    best_auroc = -1.0
    best_state = None

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc, tr_f1, tr_rec, tr_auc, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
        va_loss, va_acc, va_f1, va_rec, va_auc, _, _ = run_epoch(model, val_loader, optimizer=None)
        print(f'[{name}] ep={ep} train_auc={tr_auc:.4f} val_auc={va_auc:.4f} val_f1={va_f1:.4f}')

        if va_auc > best_auroc:
            best_auroc = va_auc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

train_model(baseline_model, 'baseline', EPOCHS_BASELINE)
train_model(strong_model, 'strong', EPOCHS_STRONG)

## 19. Inference Examples

Deployment-style inference response for binary pneumonia risk score.

In [ ]:
def infer_single(model, image_rgb):
    model.eval()
    x = val_tf(image=image_rgb)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logit = model(x).squeeze(1)
        p = float(torch.sigmoid(logit).cpu().numpy()[0])

    pred = int(p >= 0.5)
    return {
        'predicted_label': pred,
        'predicted_name': 'pneumonia' if pred == 1 else 'normal',
        'pneumonia_probability': p,
        'normal_probability': float(1.0 - p),
    }

sample = test_images[0]
resp = infer_single(strong_model, sample)
print(json.dumps(resp, indent=2))

## 20. Evaluation (AUROC + Confusion Matrix)

Main metrics: AUROC, sensitivity/recall, confusion matrix.

In [ ]:
_, b_acc, b_f1, b_rec, b_auc, by, bprob = run_epoch(baseline_model, test_loader, optimizer=None)
_, s_acc, s_f1, s_rec, s_auc, sy, sprob = run_epoch(strong_model, test_loader, optimizer=None)

bpred = [1 if p >= 0.5 else 0 for p in bprob]
spred = [1 if p >= 0.5 else 0 for p in sprob]

b_prec = precision_score(by, bpred, zero_division=0)
s_prec = precision_score(sy, spred, zero_division=0)

print('Baseline -> acc:', round(b_acc,4), 'precision:', round(b_prec,4), 'recall:', round(b_rec,4), 'f1:', round(b_f1,4), 'auroc:', round(float(b_auc),4))
print('Strong   -> acc:', round(s_acc,4), 'precision:', round(s_prec,4), 'recall:', round(s_rec,4), 'f1:', round(s_f1,4), 'auroc:', round(float(s_auc),4))

print()
print('Classification report (strong):')
print(classification_report(sy, spred, target_names=['normal','pneumonia'], zero_division=0))

## 21. Confusion Matrix and Error Analysis

In [ ]:
cm = confusion_matrix(sy, spred, labels=[0,1])
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(5,4))
sns.heatmap(cmn, annot=True, fmt='.3f', cmap='Blues', xticklabels=['normal','pneumonia'], yticklabels=['normal','pneumonia'])
plt.title('Normalized Confusion Matrix (Strong Model)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print('False negatives (pneumonia -> normal):', int(cm[1,0]))
print('False positives (normal -> pneumonia):', int(cm[0,1]))

## 22. Data Leakage Risks and Prevention

Critical leakage risks in chest X-ray tasks:

1. **Patient overlap across splits** (same patient appears in train and test)
2. **Study overlap** (multiple images from same study split across partitions)
3. **Metadata leakage** (site/device tokens, annotation artifacts)
4. **Post-processing leakage** (masks/labels accidentally encoded in pixels)

Prevention checklist:
- Split by patient ID (and study ID when available)
- Perform leakage audit before training
- Keep preprocessing identical and blind to labels
- Validate on external cohort when possible

## 23. Medical Caveats (Important)

This model is **not** a standalone diagnostic tool.

Important caveats:
1. **Clinical context missing**: symptoms, vitals, labs, and history are not included
2. **Dataset shift risk**: hospitals/devices/populations differ from training data
3. **Ground-truth uncertainty**: labels may include inter-reader variability
4. **Harm asymmetry**: false negatives can delay treatment
5. **Regulatory gap**: benchmark success is not regulatory approval

Any real use requires prospective validation, calibration, and radiologist oversight.

## 24. How To Improve

- Add external validation set
- Calibrate probabilities and tune threshold for sensitivity
- Use ensembling and uncertainty estimation
- Integrate clinical metadata in multimodal model

## 25. Production Considerations

- Monitor AUROC, recall, and false-negative rate over time
- Track drift by site/device/population
- Use human-in-the-loop review for low-confidence or high-risk predictions

## 26. Common Mistakes

- Random image-level splitting causing leakage
- Reporting only accuracy in medical triage settings
- Ignoring calibration and threshold sensitivity trade-offs

## 27. Mini Challenge / Final Summary

Mini challenge: choose threshold to maximize recall at fixed precision target and compare confusion matrices.

Summary: this notebook provides a pneumonia classifier with AUROC, confusion matrix, and explicit leakage/medical risk documentation.

In [ ]:
# Save metrics
metrics = {
    'dataset': 'RSNA/Kaggle Chest X-ray Pneumonia',
    'baseline_model': BASELINE_MODEL,
    'strong_model': STRONG_MODEL,
    'baseline_test_accuracy': float(b_acc),
    'baseline_test_precision': float(b_prec),
    'baseline_test_recall': float(b_rec),
    'baseline_test_f1': float(b_f1),
    'baseline_test_auroc': float(b_auc),
    'strong_test_accuracy': float(s_acc),
    'strong_test_precision': float(s_prec),
    'strong_test_recall': float(s_rec),
    'strong_test_f1': float(s_f1),
    'strong_test_auroc': float(s_auc),
    'false_negatives_strong': int(cm[1,0]),
    'false_positives_strong': int(cm[0,1]),
    'device': str(DEVICE),
    'synthetic_mode': bool(use_synthetic),
    'monai_available': bool(HAS_MONAI)
}

metrics_path = SAVE_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved metrics:', metrics_path)
print('Done.')